# Module 04: Attention Mechanism & Transformers from Scratch
## Master Guide: From Seq2Seq Bottlenecks to Self-Attention and the Transformer Architecture

The Transformer is the foundational architectural backbone behind nearly every breakthrough in modern AI — from **BERT**, **GPT-4**, and **Claude**, to **Llama**, **Gemini**, and multimodal vision transformers (ViT).

In this module, you will break down the Transformer step-by-step from first principles:
- Why recurrent models reached their fundamental limits.
- How the **Attention Mechanism** allows networks to dynamically focus on relevant tokens.
- The mathematical derivation of **Scaled Dot-Product Attention** and **Multi-Head Attention**.
- **Positional Encoding** and why transformers need trigonometry to understand word order.
- Complete, runnable implementation from scratch in **PyTorch**.
- Visualizing real attention weight heatmaps!

---

### 🎯 Key Questions Answered in this Module:
1. What was the "information bottleneck" of classical Encoder-Decoder Seq2Seq models?
2. What are **Queries ($Q$)**, **Keys ($K$)**, and **Values ($V$)** conceptually and mathematically?
3. Why do we divide by $\sqrt{d_k}$ in scaled dot-product attention? (The Softmax Gradient Saturation problem).
4. Why does Multi-Head Attention outperform Single-Head Attention?
5. How does Positional Encoding inject sequential order without recurrence?
6. How do Residual Connections and Layer Normalization stabilize deep transformer training?


---
## 1. The Seq2Seq Bottleneck & The Birth of Attention

Before 2015, machine translation and sequence tasks relied on **Encoder-Decoder RNNs**:
1. An **Encoder RNN** read the source sequence token by token: $x_1, x_2, \dots, x_T$.
2. The final hidden state $h_T$ was designated as the **Context Vector ($c$)**.
3. A **Decoder RNN** generated the target sequence conditioned *solely* on this single context vector $c$.

```text
[Input]  The   quick   brown   fox   jumps   over   the   lazy   dog
           |       |       |     |       |      |     |      |      |
         [RNN] -> [RNN] -> ... -> ... -> ... -> ... -> ... -> ... -> [RNN] -> [Context Vector c]
                                                                                       |
                                                                             [Decoder Generates Output]
```

### 🚨 The Problem:
Imagine reading a 100-page book and being forced to summarize its entire contents into a single 256-dimensional numerical vector, and then having someone else translate the entire book into French based *only* on that vector!
- Information from early words decays or gets overwritten.
- Performance drops drastically on sentences longer than 20 words.

### 💡 The Solution: Attention (Bahdanau et al., 2014)
Instead of forcing the encoder to compress everything into one fixed vector:
1. Retain **all** encoder hidden states: $(h_1, h_2, \dots, h_T)$.
2. At each decoding step $t$, let the decoder dynamically look back at all encoder states and assign an **importance weight (attention score)** $\alpha_{ti}$ to each word.
3. Compute a dynamic context vector as a **weighted average**:
$$c_t = \sum_{i=1}^T \alpha_{ti} h_i$$


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")


---
## 2. The Transformer Paradigm: "Attention Is All You Need" (2017)

While Attention improved RNNs, RNNs retained a fundamental flaw: **Sequential Dependency**.
Because $h_t$ strictly requires $h_{t-1}$, **RNNs cannot be parallelized across time steps!** Training on millions of tokens was painfully slow.

Vaswani et al. (2017) proposed the radical idea: **Discard recurrence completely. Rely solely on Attention!**

### The Query, Key, Value ($Q, K, V$) Intuition
Think of Attention like searching on YouTube or Google:
- **Query ($Q$)**: The search string you type into the search bar: *"PyTorch LSTM tutorial"*.
- **Keys ($K$)**: The title, tags, and metadata of all candidate videos in the database.
- **Values ($V$)**: The actual video content itself.

The search engine compares your **Query** against all **Keys** using similarity (dot product). The videos whose keys best match your query get the highest attention weight, and you receive a weighted blend of their **Values**!

```text
             Query (Q)       Keys (K)
                 \              /
                  \            /
                [ Dot Product (Q @ K^T) ]  --> Raw similarity scores
                           |
                [ Scale by sqrt(d_k) ]     --> Stabilize variance
                           |
                [ Softmax Normalization ]  --> Attention weights (probabilities sum to 1.0)
                           |
                           v
                [ Weights @ Values (V) ]   --> Weighted combination of information!
```


---
## 3. Mathematical Deep-Dive: Scaled Dot-Product Attention

The formal equation for Scaled Dot-Product Attention is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

Where:
- $Q \in \mathbb{R}^{N \times d_k}$: $N$ queries of dimension $d_k$
- $K \in \mathbb{R}^{M \times d_k}$: $M$ keys of dimension $d_k$
- $V \in \mathbb{R}^{M \times d_v}$: $M$ values of dimension $d_v$
- $Q K^T \in \mathbb{R}^{N \times M}$: Pairwise dot products measuring compatibility
- $\sqrt{d_k}$: The scaling factor

### 🔬 Why do we divide by $\sqrt{d_k}$? (The Softmax Gradient Proof)
Assume that components of $q$ and $k$ are independent random variables with mean 0 and variance 1:
$$\mathbb{E}[q_i] = 0, \quad \text{Var}(q_i) = 1, \quad \mathbb{E}[k_i] = 0, \quad \text{Var}(k_i) = 1$$

Their dot product is:
$$z = q \cdot k = \sum_{i=1}^{d_k} q_i k_i$$

The expected value is $\mathbb{E}[z] = 0$, but the variance is:
$$\text{Var}(z) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = \sum_{i=1}^{d_k} \text{Var}(q_i)\text{Var}(k_i) = d_k$$

As the dimension $d_k$ grows large (e.g., $d_k = 64$ or $128$):
- The magnitude of dot products grows large: standard deviation is $\sqrt{d_k}$.
- When fed into the $\text{softmax}(z_i) = \frac{e^{z_i}}{\sum e^{z_j}}$, large values push the softmax function into regions where its derivative $\frac{\partial \text{softmax}}{\partial z}$ is **virtually zero**!
- By dividing by $\sqrt{d_k}$, we scale the variance back down to $1.0$, preserving healthy gradient flow!


In [ ]:
# Numerical proof of Softmax Saturation without scaling
d_k_values = [8, 32, 64, 128, 512]
softmax_max_probs_unscaled = []
softmax_max_probs_scaled = []

for d in d_k_values:
    # Random query and key vectors of dimension d
    q = np.random.randn(1, d)
    k = np.random.randn(10, d)
    
    # Unscaled dot product
    dots_unscaled = np.dot(q, k.T)
    # Scaled dot product
    dots_scaled = dots_unscaled / np.sqrt(d)
    
    # Softmax function
    sm_unscaled = np.exp(dots_unscaled - np.max(dots_unscaled))
    sm_unscaled = sm_unscaled / sm_unscaled.sum()
    
    sm_scaled = np.exp(dots_scaled - np.max(dots_scaled))
    sm_scaled = sm_scaled / sm_scaled.sum()
    
    softmax_max_probs_unscaled.append(np.max(sm_unscaled))
    softmax_max_probs_scaled.append(np.max(sm_scaled))

plt.figure(figsize=(9, 4))
plt.plot(d_k_values, softmax_max_probs_unscaled, 'o--', color='crimson', lw=2, label="Unscaled Dot Product")
plt.plot(d_k_values, softmax_max_probs_scaled, 's-', color='teal', lw=2, label="Scaled by sqrt(d_k)")
plt.xlabel("Key Dimension (d_k)")
plt.ylabel("Maximum Probability in Softmax")
plt.title("Effect of sqrt(d_k) Scaling on Softmax Saturation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 4. Coding Scaled Dot-Product Attention from Scratch in PyTorch

Let's implement a modular `ScaledDotProductAttention` class that supports:
1. Batch computation: $(B, \text{heads}, \text{seq\_len}, d_k)$
2. Optional Attention Mask (used in decoder causal masking or padding masking)
3. Dropout on attention weights


In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else None
        
    def forward(self, Q, K, V, mask=None):
        """
        Q: (batch_size, n_heads, seq_len_q, d_k)
        K: (batch_size, n_heads, seq_len_k, d_k)
        V: (batch_size, n_heads, seq_len_v, d_v) where seq_len_k == seq_len_v
        mask: optional tensor of shape (batch_size, 1, 1, seq_len_k) or (batch_size, 1, seq_len_q, seq_len_k)
        """
        d_k = Q.size(-1)
        
        # 1. Compute dot products Q @ K^T
        # scores shape: (batch_size, n_heads, seq_len_q, seq_len_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        # 2. Apply optional mask (e.g. causal mask or padding mask)
        if mask is not None:
            # Set masked positions to -inf so softmax makes them 0
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # 3. Softmax over the key dimension
        attn_weights = F.softmax(scores, dim=-1)
        
        # 4. Optional attention dropout
        if self.dropout is not None:
            attn_weights = self.dropout(attn_weights)
            
        # 5. Weighted combination with values V
        # output shape: (batch_size, n_heads, seq_len_q, d_v)
        output = torch.matmul(attn_weights, V)
        
        return output, attn_weights

# Verify ScaledDotProductAttention
batch_size, n_heads, seq_len, d_k = 2, 4, 6, 16
dummy_Q = torch.randn(batch_size, n_heads, seq_len, d_k)
dummy_K = torch.randn(batch_size, n_heads, seq_len, d_k)
dummy_V = torch.randn(batch_size, n_heads, seq_len, d_k)

attention_module = ScaledDotProductAttention()
out, weights = attention_module(dummy_Q, dummy_K, dummy_V)

print(f"Output shape: {out.shape} -> Expected (2, 4, 6, 16)")
print(f"Weights shape: {weights.shape} -> Expected (2, 4, 6, 6)")
print(f"Row sum of weights (must equal 1.0): {weights[0, 0, 0].sum().item():.4f}")


---
## 5. Multi-Head Attention: Exploring Multiple Representation Subspaces

Instead of performing a single attention function with $d_{model}$-dimensional queries, keys, and values, the Transformer linearly projects $Q$, $K$, and $V$ into $h$ different subspaces of dimension $d_k = d_{model} / h$:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \text{head}_2, \dots, \text{head}_h) W^O$$
$$\text{where} \quad \text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

### Why Multi-Head?
If you have only 1 attention head, the model can only focus on one dominant relationship per word.
With $h=8$ heads:
- **Head 1**: Tracks subject-verb relations (*The **chef** ... **cooks***)
- **Head 2**: Tracks pronoun antecedents (***Alice** told Bob **she** was leaving*)
- **Head 3**: Tracks local adjacent modifiers (***very fast*** car)
- **Head 4**: Tracks syntactic dependencies and punctuation!


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads!"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projection matrices
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Final output projection matrix
        self.W_o = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(dropout=dropout)
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. Project inputs: (batch, seq_len, d_model)
        Q = self.W_q(q)
        K = self.W_k(k)
        V = self.W_v(v)
        
        # 2. Reshape into (batch, num_heads, seq_len, d_k)
        # Note: transpose(1, 2) swaps seq_len and num_heads
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Apply Scaled Dot-Product Attention
        context, attn_weights = self.attention(Q, K, V, mask=mask)
        
        # 4. Concatenate heads back into (batch, seq_len, d_model)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. Final linear projection
        output = self.W_o(context)
        return output, attn_weights

# Test MultiHeadAttention
mha = MultiHeadAttention(d_model=64, num_heads=8)
x_in = torch.randn(2, 10, 64) # batch 2, seq_len 10, d_model 64
out, weights = mha(x_in, x_in, x_in) # Self-Attention (Q=K=V=x)

print(f"MultiHead Attention Output shape: {out.shape} -> Expected (2, 10, 64)")
print(f"MultiHead Attention Weights shape: {weights.shape} -> Expected (2, 8, 10, 10)")


---
## 6. Positional Encoding: Giving the Transformer a Sense of Order

In pure self-attention, the operation is **permutation equivariant**: if you shuffle the input words, the attention weights and outputs simply shuffle identically!
Without positional encoding, the sentences:
- *"The dog bit the cat"*
- *"The cat bit the dog"*
would have identical internal calculations!

Because the Transformer contains no recurrence or convolution, we must inject positional information directly into the input embeddings:
$$\text{Input} = \text{Word Embedding} + \text{Positional Encoding}$$

Vaswani et al. introduced **Sinusoidal Positional Encoding**:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Where:
- $pos$ is the token's position in the sequence ($0, 1, 2, \dots, T-1$)
- $i$ is the dimension index ($0 \le i < d_{model}/2$)

### Why Sinusoidal Waves?
1. **Bounded values**: Always in range $[-1, +1]$.
2. **Deterministic & extrapolation**: Can generalize to sequence lengths longer than seen during training.
3. **Linear relative offset**: By the trigonometric addition theorem:
   $$\sin(\alpha + \beta) = \sin \alpha \cos \beta + \cos \alpha \sin \beta$$
   The encoding for position $pos + k$ can be represented as a fixed linear transformation of the encoding at position $pos$! This allows the model to easily learn relative distances between words!


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        
        # Create positional encoding matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # (max_len, 1)
        
        # Division term: 10000^(2i / d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices, cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe) # Persisted but not updated via backprop
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

# Visualize Positional Encodings
pos_enc = PositionalEncoding(d_model=64, max_len=100)
sample_pe = pos_enc.pe[0, :50, :].numpy()

plt.figure(figsize=(10, 5))
plt.imshow(sample_pe, cmap='viridis', aspect='auto')
plt.title("Sinusoidal Positional Encoding Matrix (First 50 tokens x 64 dims)")
plt.xlabel("Embedding Dimension (0 to 63)")
plt.ylabel("Token Position in Sequence (0 to 49)")
plt.colorbar(label="Value")
plt.tight_layout()
plt.show()


---
## 7. Transformer Encoder Block: Putting It All Together

A complete **Transformer Encoder Layer** combines:
1. **Multi-Head Self-Attention**
2. **Residual Connection & Layer Normalization**: $\text{LayerNorm}(x + \text{MultiHead}(x))$
3. **Position-wise Feed-Forward Network**: Two linear layers with a ReLU/GELU activation:
   $$\text{FFN}(x) = \max(0, x W_1 + b_1) W_2 + b_2$$
4. Second **Residual Connection & Layer Normalization**: $\text{LayerNorm}(x + \text{FFN}(x))$

```text
               Input Embeddings + Positional Encoding
                                |
                                v
                   +--------> [ + ] <---------+
                   |            |             | (Residual Connection)
                   |      [ LayerNorm ]       |
                   |            |             |
                   |   [ Multi-Head Attn ]    |
                   |            ^             |
                   +------------+-------------+
                                |
                   +--------> [ + ] <---------+
                   |            |             | (Residual Connection)
                   |      [ LayerNorm ]       |
                   |            |             |
                   |       [ FeedForward ]    |
                   |            ^             |
                   +------------+-------------+
                                |
                                v
                         Layer Output
```


In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model=64, num_heads=8, d_ff=256, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # 1. Multi-head self-attention with residual and norm
        attn_out, attn_weights = self.mha(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # 2. Feed-forward with residual and norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        
        return x, attn_weights

# Test full encoder layer
enc_layer = TransformerEncoderLayer(d_model=64, num_heads=8, d_ff=256)
dummy_tokens = torch.randn(2, 12, 64)
enc_out, weights = enc_layer(dummy_tokens)

print(f"Encoder Output Shape: {enc_out.shape} -> Expected (2, 12, 64)")
print(f"Attention Weights Shape: {weights.shape} -> Expected (2, 8, 12, 12)")


---
## 8. Visualizing Self-Attention Heatmaps

One of the greatest advantages of Attention is **interpretability**. By inspecting the attention weights matrix $\alpha \in \mathbb{R}^{T \times T}$, we can see exactly which words attend to which other words in a sentence!


In [ ]:
# Sample sentence demonstration
tokens = ["The", "animal", "didn't", "cross", "the", "street", "because", "it", "was", "too", "tired"]
seq_l = len(tokens)

# Let's visualize an illustrative attention pattern for the pronoun 'it'
# In language, 'it' should attend strongly back to 'animal'!
simulated_weights = np.zeros((seq_l, seq_l))

for i in range(seq_l):
    simulated_weights[i, i] = 0.4  # Self focus
    if i > 0:
        simulated_weights[i, i-1] = 0.25 # Adjacent word focus
    if i < seq_l - 1:
        simulated_weights[i, i+1] = 0.15

# Focus for pronoun 'it' (index 7) -> connects to 'animal' (index 1)
simulated_weights[7, 1] = 0.65
simulated_weights[7, 7] = 0.20
simulated_weights[7, :] = simulated_weights[7, :] / simulated_weights[7, :].sum()

# Plot Attention Heatmap
plt.figure(figsize=(8, 7))
plt.imshow(simulated_weights, cmap='Blues', interpolation='nearest')
plt.xticks(range(seq_l), tokens, rotation=45, ha='right', fontsize=11)
plt.yticks(range(seq_l), tokens, fontsize=11)
plt.xlabel("Key Tokens (What is being attended to)", fontsize=12)
plt.ylabel("Query Tokens (What is attending)", fontsize=12)
plt.title("Self-Attention Map: Notice pronoun 'it' attending strongly to 'animal'!", fontsize=13)
plt.colorbar(label="Attention Weight")
plt.tight_layout()
plt.show()


---
## 9. Self-Assessment Practice Exercises & Detailed Solutions

### Exercise 1: Complexity Analysis (RNN vs Transformer)
For a sequence of length $n$ with representation dimension $d$:
1. What is the computational complexity per layer of a Standard RNN?
2. What is the computational complexity per layer of Self-Attention?
3. In what regime (value of $n$ vs $d$) is Self-Attention computationally faster than an RNN?

---
### Exercise 2: Causal Masking
In an autoregressive decoder model (like GPT-2 or Llama), why is a "Causal Attention Mask" strictly required? What would happen during generation if you didn't apply this mask?

---
### Solutions:
<details>
<summary><b>Click to expand Solution 1</b></summary>

1. **RNN Complexity**: $O(n \cdot d^2)$ operations per layer (for each of the $n$ steps, a matrix-vector product of size $d \times d$). Sequential operations: $O(n)$ (cannot be parallelized).
2. **Self-Attention Complexity**: $O(n^2 \cdot d)$ operations per layer ($Q K^T$ involves multiplying $(n \times d) \times (d \times n) = n^2 \cdot d$). Sequential operations: $O(1)$ (fully parallel!).
3. **Regime**: When $n < d$ (which is very common for standard sentence lengths like $n=128$ and $d=768$), Self-Attention has **lower computational complexity** AND can be executed in parallel on GPUs! (However, for ultra-long documents where $n \gg d$, the $n^2$ quadratic cost becomes the bottleneck, motivating FlashAttention and linear attention variants).
</details>

<details>
<summary><b>Click to expand Solution 2</b></summary>

In language modeling / text generation, the task is predicting token $x_{t+1}$ given only tokens $(x_1, \dots, x_t)$.  
If we did not mask future positions ($t+1, \dots, T$) with $-\infty$, the model would "cheat" by looking ahead at the future tokens it is supposed to predict. The causal upper-triangular mask ensures position $i$ can only attend to positions $j \le i$.
</details>


---
## 10. Summary & Next Steps
1. **Attention** eliminated the fixed-vector information bottleneck by providing dynamic, soft alignment over all input tokens.
2. **Transformers** dispensed with recurrence completely, enabling massive parallelization across tokens on modern hardware.
3. **Scaled Dot-Product Attention** uses $\frac{1}{\sqrt{d_k}}$ to prevent softmax gradient saturation.
4. **Multi-Head Attention** allows the model to attend to multiple aspects (syntax, semantics, coreference) simultaneously.
5. **Positional Encodings** inject sequential awareness into permutation-equivariant attention.
6. In **Module 05**, we will explore how pre-trained Transformers revolutionized NLP through **BERT**, **RoBERTa**, and the **Hugging Face** ecosystem, and fine-tune a state-of-the-art transformer on real data!
